# Exploración de datos de mi proyecto: demanda M5, tienda CA_1

Archivo: `OBT.OBT_SALES` del pipeline M5 (Walmart), filtrada a la tienda CA_1 (`../data/processed/obt_sales_CA_1.csv.gz`). Objetivo: `units` (unidades vendidas del producto en la tienda ese día).

In [1]:
import os

import matplotlib.pyplot as plt
import pandas as pd

os.makedirs("figuras", exist_ok=True)
datos = pd.read_csv("../data/processed/obt_sales_CA_1.csv.gz")
OBJETIVO = "units"

## Paso 1: resumen

In [2]:
print("Filas y columnas:", datos.shape)
print(datos.isna().sum()[datos.isna().sum() > 0])
print(datos[[OBJETIVO, "sell_price", "revenue"]].describe().round(2))

Filas y columnas: (5918109, 40)


sell_price         1129842
revenue            1129842
event_name_1       5436367
event_type_1       5436367
event_name_2       5905913
event_type_2       5905913
date_key_ly_364    1109836
dtype: int64


            units  sell_price     revenue
count  5918109.00  4788267.00  4788267.00
mean         1.32        4.41        4.79
std          4.05        3.40       11.68
min          0.00        0.01        0.00
25%          0.00        2.22        0.00
50%          0.00        3.47        0.00
75%          1.00        5.77        5.91
max        648.00       30.98     2164.32


5 918 109 filas y 40 columnas (3 049 productos × 1 941 días). Faltantes: `sell_price` y `revenue` (1 129 842, las filas pre-lanzamiento), `date_key_ly_364` (el primer año), `event_*` (días sin evento, un nulo legítimo).

## Paso 2: distribución de la variable objetivo

In [3]:
plt.figure()
plt.hist(datos[OBJETIVO], bins=40)
plt.title(OBJETIVO)
plt.savefig("figuras/distribucion_objetivo.png")
plt.show()

`units`: mediana 0, Q3 1, máximo 648: muy asimétrica, con muchos ceros y una cola larga.

## Paso 3: atípicos con la regla del IQR

In [4]:
q1, q3 = datos[OBJETIVO].quantile([0.25, 0.75])
iqr = q3 - q1
fuera = datos[(datos[OBJETIVO] < q1 - 1.5 * iqr) | (datos[OBJETIVO] > q3 + 1.5 * iqr)]
print(f"Atipicos de {OBJETIVO} segun IQR: {len(fuera)} ({len(fuera) / len(datos):.1%})")

Atipicos de units segun IQR: 839509 (14.2%)


El IQR marca 839 509 atípicos (14.2%): todo valor ≥ 3. Con Q1 = 0 y Q3 = 1 la regla no sirve; solo refleja que el objetivo es asimétrico.

## Paso 4: relación con el departamento

In [5]:
por_depto = datos.groupby("dept_id")[OBJETIVO].mean().sort_values()
print(por_depto.round(2))
por_depto.plot.barh()
plt.title("units promedio por departamento")
plt.tight_layout()
plt.savefig("figuras/objetivo_por_departamento.png")
plt.show()

dept_id
HOBBIES_2      0.20
HOUSEHOLD_2    0.35
HOBBIES_1      1.03
HOUSEHOLD_1    1.09
FOODS_2        1.17
FOODS_1        1.38
FOODS_3        2.50
Name: units, dtype: float64


FOODS_3 vende 2.50 unidades por día y producto, 12 veces más que HOBBIES_2 (0.20): el departamento define el nivel de la serie, así que el modelo necesita `dept_id`.

## Paso 5: tres hallazgos que afectan el modelado

In [6]:
pre = datos.query("is_pre_launch")
print(f"Filas pre-lanzamiento: {len(pre)} ({len(pre) / len(datos):.1%})")
print(f"Ceros de units: {(datos[OBJETIVO] == 0).mean():.1%} del total, {(datos.query('not is_pre_launch')[OBJETIVO] == 0).mean():.1%} sin pre-lanzamiento")

Filas pre-lanzamiento: 1129842 (19.1%)

Ceros de units: 63.8% del total, 55.2% sin pre-lanzamiento


**Hallazgo 1.** El 19.1% de las filas son días antes de que el producto se lanzara: `units` siempre es 0 y `sell_price` es nulo. Son ceros falsos (suben los ceros de 55.2% a 63.8%). **Cambio:** quitar las filas con `is_pre_launch` antes de entrenar.

In [7]:
vendibles = datos.query("not is_pre_launch")
ceros = vendibles.query(f"{OBJETIVO} == 0")
print(f"Ceros marcados como quiebre (conservador): {ceros['is_suspected_stockout_conservative'].mean():.1%}")
print(f"Ceros marcados como quiebre (binomial negativa): {ceros['is_suspected_stockout_nb'].mean():.1%}")
print(vendibles.groupby("is_christmas_closed")[OBJETIVO].agg(["size", "mean"]).round(2))

Ceros marcados como quiebre (conservador): 9.5%
Ceros marcados como quiebre (binomial negativa): 26.5%
                        size  mean
is_christmas_closed               
False                4775625  1.64
True                   12642  0.00


**Hallazgo 2.** No todo cero es falta de demanda: entre 9.5% y 26.5% de los ceros son posibles quiebres de stock, y los registros de Navidad (tienda cerrada) tienen 0. **Cambio:** excluir la Navidad y quitar o ponderar a la baja los quiebres; además, por ser un conteo sesgado, conviene una pérdida Poisson/Tweedie o `log1p(units)`.

In [8]:
con_precio = vendibles.dropna(subset=["sell_price"])
print(f"revenue == units * sell_price: {((con_precio['revenue'] - con_precio[OBJETIVO] * con_precio['sell_price']).abs() < 0.01).mean():.1%}")
print(vendibles[[OBJETIVO, "revenue", "sell_price", "is_snap", "is_event_day", "is_weekend"]].corr()[OBJETIVO].round(3))

revenue == units * sell_price: 100.0%


units           1.000
revenue         0.730
sell_price     -0.160
is_snap         0.017
is_event_day   -0.006
is_weekend      0.062
Name: units, dtype: float64


**Hallazgo 3.** `revenue` es exactamente `units × sell_price` y tiene correlación 0.73 con el objetivo: es fuga de información (no se conoce antes de vender). `_loaded_at` y `week_idx` son columnas de linaje del pipeline. **Cambio:** quitarlas de las variables de entrada; `sell_price` sí se puede usar.